# Interpreto Other Methods Tutorial

This notebook demonstrates the use of specialized interpretability methods in Interpreto that provide unique insights into model behavior.

Currently, this includes the **Logit Lens** method, which shows what predictions models make at intermediate layers.


## Available Methods

**Layer-wise Analysis Methods:**
- **Logit Lens**: Shows intermediate predictions by applying the final prediction head to intermediate activations
  - Language Models: Token predictions at each layer
  - Classification Models: Class predictions at each layer

## Imports

In [ ]:
import sys
sys.path.append("../..")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification

from interpreto import ModelWithSplitPoints, LogitLens

## 1. Logit Lens with Language Models

The Logit Lens reveals what tokens a language model is "thinking" about at each layer. This helps understand how the model's understanding evolves through the layers.

### Setup Language Model

In [ ]:
# Load a small language model for demonstration
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Set up padding token if not available
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Create model with split points for intermediate layer analysis
model_with_split_points = ModelWithSplitPoints(
    model_name,
    split_points=["transformer.h.0.mlp.c_proj", "transformer.h.3.mlp.c_proj", "transformer.h.6.mlp.c_proj", "transformer.h.11.mlp.c_proj"],
    model_autoclass="AutoModelForCausalLM",
    device_map="cuda"
)
print(model_with_split_points)
print(f"Model loaded: {model_name}")
print(f"Split points: {model_with_split_points.split_points}")

### Create Logit Lens Instance

In [ ]:
# Create LogitLens instance
lens = LogitLens(
    model_with_split_points, 
    tokenizer,
    nb_token=5,  # Show top-5 token predictions
    batch_size=4,
)

print(f"LogitLens initialized for {lens.__class__.__name__}")
print(f"Model head: {lens.head_name}")
print(f"Features dimension: {lens.features_dim}")

### Analyze Text with Logit Lens

In [ ]:
# Example text for analysis
text = "The cat sat on the"

print(f"Analyzing text: '{text}'")
print("\nTokens:", tokenizer.tokenize(text))

In [ ]:
# Get explanations (raw data)
explanations = lens(text)

# Show structure of results
print("Layers analyzed:", list(explanations.keys()))
print("\nExample data structure for first layer:")
first_layer = list(explanations.keys())[0]
print(f"Layer '{first_layer}':")
print(f"  - Tokens shape: {explanations[first_layer]['tokens'].shape}")
print(f"  - Probabilities shape: {explanations[first_layer]['proba'].shape}")

### Interactive Visualization

In [ ]:
# Interactive visualization
lens.lens(text)

### Multiple Examples

In [ ]:
# Analyze multiple sentences
texts = [
    "The weather is",
    "Machine learning is",
    "Paris is the capital of"
]

for text in texts:
    print(f"\n--- Analyzing: '{text}' ---")
    lens.lens(text)

## 2. Logit Lens with Classification Models

For classification models, the Logit Lens shows what classes the model is leaning towards at each layer.

### Setup Classification Model

In [ ]:
# Load a sentiment classification model
model_name_cls = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer_cls = AutoTokenizer.from_pretrained(model_name_cls)

# Create model with split points for classification
model_with_split_points_cls = ModelWithSplitPoints(
    model_name_cls,
    split_points=[
        "roberta.encoder.layer.3.output.dense", 
        "roberta.encoder.layer.7.output.dense", 
        "roberta.encoder.layer.11.output.dense"
    ],
    model_autoclass="AutoModelForSequenceClassification",
)

print(f"Classification model loaded: {model_name_cls}")
print(f"Split points: {model_with_split_points_cls.split_points}")

### Create Classification Logit Lens

In [ ]:
# Create LogitLens for classification
lens_cls = LogitLens(
    model_with_split_points_cls,
    tokenizer_cls,
    pooling_strategy="cls",  # Use [CLS] token for classification
    batch_size=4,
)

print(f"Classification LogitLens initialized")
print(f"Model head: {lens_cls.head_name}")
print(f"Number of classes: {lens_cls.num_classes}")
print(f"Class labels: {lens_cls._get_output_labels()}")

### Analyze Sentiment Classification

In [ ]:
# Example texts with different sentiments
sentiment_texts = [
    "I love this movie! It's absolutely fantastic.",
    "This product is okay, nothing special.",
    "I hate this service, it's terrible and slow."
]

for text in sentiment_texts:
    print(f"\n--- Sentiment Analysis: '{text}' ---")
    
    # Get raw explanations
    explanations = lens_cls(text)
    
    # Show predictions for each layer
    for layer_name, results in explanations.items():
        predicted_label = results['predicted_labels'][0]
        confidence = results['confidence_scores'][0]
        print(f"Layer {layer_name}: {predicted_label} (confidence: {confidence:.3f})")
    
    # Interactive visualization
    lens_cls.lens(text)

## 3. Advanced Usage

### Custom Layer Selection

In [ ]:
# Analyze only specific layers
text = "The future of AI is"
specific_layers = ["transformer.h.3.mlp.c_proj", "transformer.h.11.mlp.c_proj"]

print(f"Analyzing '{text}' at specific layers: {specific_layers}")
explanations = lens(text, layers_name=specific_layers)

print("Analyzed layers:", list(explanations.keys()))
lens.lens(text, layers_name=specific_layers)

### Batch Processing

In [ ]:
# Process multiple texts efficiently
batch_texts = [
    "The sun is shining",
    "Technology advances rapidly", 
    "Music brings joy",
    "Learning never stops"
]

print(f"Batch processing {len(batch_texts)} texts...")
batch_explanations = lens(batch_texts)

print("Batch processing complete!")
print(f"Processed {len(batch_texts)} texts across {len(batch_explanations)} layers")

# Visualize batch results
lens.lens(batch_texts)

## 4. Configuration Options

### Normalization Effects

In [ ]:
# Compare with and without normalization
text = "The quick brown fox"

# Without normalization
lens_no_norm = LogitLens(
    model_with_split_points,
    tokenizer,
    nb_token=3,
    normalization=False
)

# With normalization
lens_with_norm = LogitLens(
    model_with_split_points,
    tokenizer,
    nb_token=3,
    normalization=True
)

print("=== Without Normalization ===")
lens_no_norm.lens(text)

print("\n=== With Normalization ===")
lens_with_norm.lens(text)

### Different Token Counts

In [ ]:
# Show different numbers of top tokens
text = "Artificial intelligence will"

for nb_tokens in [1, 3, 10]:
    print(f"\n=== Top-{nb_tokens} Tokens ===")
    lens_custom = LogitLens(
        model_with_split_points,
        tokenizer,
        nb_token=nb_tokens
    )
    lens_custom.lens(text)